In [2]:
import os
import re
import cv2
import time
import math
from  glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip -q install ultralytics

from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict

import xml.etree.ElementTree as ET
import shutil

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


# Dataset

In [3]:
# ================================================
# CONFIGURATION
# Classe finale
FINAL_CLASSES = ["car"]

# UA-DETRAC types autorisés
KEEP_TYPES = ["car", "van"]   # fusionnés
IGNORE_TYPES = ["bus", "others"]

# Paths
#ann_train = "/kaggle/input/ua-detrac-orig/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML"
ann_val   = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"
img_root  = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Images/DETRAC-Images"

out_images_train = "data/images/train"
out_labels_train = "data/labels/train"
out_images_val   = "data/images/val"
out_labels_val   = "data/labels/val"

for d in [
    out_images_train, out_labels_train,
    out_images_val, out_labels_val
]:
    os.makedirs(d, exist_ok=True)

# ==================================================
# CLASS MAPPING
# ==================================================

def map_vehicle_type(vtype):
    """
    car  -> 0
    van  -> 0
    bus / others -> ignored
    """
    if vtype in KEEP_TYPES:
        return 0
    return None

# ==================================================
# CONVERSION BOX
# ==================================================

def convert_to_yolo(left, top, width, height, img_w, img_h):
    xmin = float(left)
    ymin = float(top)
    xmax = xmin + float(width)
    ymax = ymin + float(height)

    return (
        ((xmin + xmax) / 2) / img_w,
        ((ymin + ymax) / 2) / img_h,
        (xmax - xmin) / img_w,
        (ymax - ymin) / img_h
    )

# ==================================================
# XML PROCESSING
# ==================================================

def process_xml(xml_path, mode="train"):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    seq_name = root.attrib["name"]
    img_dir = os.path.join(img_root, seq_name)

    out_img_dir = out_images_train if mode == "train" else out_images_val
    out_lbl_dir = out_labels_train if mode == "train" else out_labels_val

    for frame in root.findall("frame"):
        frame_num = int(frame.attrib["num"])

        # Ici, on garde toutes les frames
        img_name = f"img{frame_num:05d}.jpg"
        img_path = os.path.join(img_dir, img_name)

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        target_list = frame.find("target_list")
        label_lines = []

        if target_list is not None:
            for obj in target_list.findall("target"):
                attr = obj.find("attribute")
                if attr is None:
                    continue

                vtype = attr.attrib.get("vehicle_type")
                cls_id = map_vehicle_type(vtype)

                if cls_id is None:
                    continue

                box = obj.find("box")
                if box is None:
                    continue

                x, y, bw, bh = convert_to_yolo(
                    box.attrib["left"],
                    box.attrib["top"],
                    box.attrib["width"],
                    box.attrib["height"],
                    w, h
                )

                label_lines.append(
                    f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}"
                )

        out_name = f"{seq_name}_img{frame_num:05d}"

        # écrire label même vide
        with open(os.path.join(out_lbl_dir, out_name + ".txt"), "w") as f:
            if label_lines:
                f.write("\n".join(label_lines))

        dst_img = os.path.join(out_img_dir, out_name + ".jpg")
        if not os.path.exists(dst_img):
            shutil.copy(img_path, dst_img)

# ==================================================
# EXECUTION
# ==================================================

print("Conversion VAL sans frame sampling...")
for xml_file in glob(os.path.join(ann_val, "*.xml")):
    process_xml(xml_file, mode="val")

print("Dataset YOLO UA-DETRAC prêt sans frame stride.")

Conversion VAL sans frame sampling...
Dataset YOLO UA-DETRAC prêt sans frame stride.


# Calibration 

In [4]:
DATASET_DIR = Path("/kaggle/working/data")

IMG_VAL_DIR = DATASET_DIR / "images" / "val"
LBL_VAL_DIR = DATASET_DIR / "labels" / "val"

MODEL_PATH = "/kaggle/input/models/chaymaeramii/modelname/other/default/1/best.pt"

# Car-only model
NUM_CLASSES = 1
CLASS_NAMES = {0: "car"}

IMG_SIZE = 768
CONF_THRES = 0.30     
IOU_NMS = 0.60

# Event / adaptive
K_MIN = 1
K_MAX = 5
N_MAX_CARS = 20

# Pour ne pas overfitter les seuils sur toute la validation
CALIBRATION_SEQUENCE_RATIO = 0.20

RESULTS_DIR = Path("/kaggle/working/ua_detrac_event_adaptive_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Images val:", IMG_VAL_DIR)
print("Labels val:", LBL_VAL_DIR)
print("Model:", MODEL_PATH)
print("Results:", RESULTS_DIR)

Images val: /kaggle/working/data/images/val
Labels val: /kaggle/working/data/labels/val
Model: /kaggle/input/models/chaymaeramii/modelname/other/default/1/best.pt
Results: /kaggle/working/ua_detrac_event_adaptive_results


In [5]:
def list_images_recursive(img_dir):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
    files = []
    for ext in exts:
        files.extend(img_dir.rglob(ext))
    return sorted(files)


def image_id(path):
    return Path(path).stem


def extract_frame_number(path):
    stem = Path(path).stem
    nums = re.findall(r"\d+", stem)
    if len(nums) == 0:
        return 0
    return int(nums[-1])


def infer_sequence_id(path, root_dir):
    """
    UA-DETRAC est normalement organisé par séquences MVI_xxxx.
    Si les images sont dans des dossiers, le parent devient la séquence.
    Sinon, on essaie d'extraire MVI_xxxx du nom.
    """
    path = Path(path)
    rel = path.relative_to(root_dir)

    if len(rel.parts) > 1:
        return rel.parts[0]

    stem = path.stem

    m = re.search(r"(MVI[_-]?\d+)", stem, re.IGNORECASE)
    if m:
        return m.group(1)

    # fallback : préfixe avant le dernier numéro
    seq = re.sub(r"[_-]?\d+$", "", stem)
    if seq == "":
        seq = "single_stream"

    return seq


def build_sequences(img_dir):
    images = list_images_recursive(img_dir)
    sequences = defaultdict(list)

    for p in images:
        sid = infer_sequence_id(p, img_dir)
        sequences[sid].append(p)

    ordered_sequences = {}
    for sid, paths in sequences.items():
        ordered_sequences[sid] = sorted(paths, key=extract_frame_number)

    return ordered_sequences


sequences = build_sequences(IMG_VAL_DIR)

print("Nombre de séquences:", len(sequences))
print("Nombre total images:", sum(len(v) for v in sequences.values()))

for sid, paths in list(sequences.items())[:5]:
    print(sid, ":", len(paths), "frames", "ex:", paths[0].name)

Nombre de séquences: 40
Nombre total images: 56167
MVI_39031 : 1470 frames ex: MVI_39031_img00001.jpg
MVI_39051 : 1051 frames ex: MVI_39051_img00001.jpg
MVI_39211 : 1566 frames ex: MVI_39211_img00001.jpg
MVI_39271 : 1566 frames ex: MVI_39271_img00001.jpg
MVI_39311 : 1505 frames ex: MVI_39311_img00001.jpg


In [6]:
def load_image_shape(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Image non lisible: {img_path}")
    h, w = img.shape[:2]
    return h, w



def find_label_path(img_path, img_root, label_root):
    """
    Cherche le .txt correspondant à une image.
    Supporte :
    - labels/val/sequence/frame.txt
    - labels/val/frame.txt
    """
    img_path = Path(img_path)
    rel = img_path.relative_to(img_root)

    candidate1 = label_root / rel.with_suffix(".txt")
    if candidate1.exists():
        return candidate1

    candidate2 = label_root / f"{img_path.stem}.txt"
    if candidate2.exists():
        return candidate2

    return candidate1



def yolo_label_to_xyxy(label_path, img_w, img_h):
    """
    YOLO label:
    cls xc yc w h
    Retour:
    [cls, x1, y1, x2, y2]
    """
    boxes = []

    if not Path(label_path).exists():
        return np.zeros((0, 5), dtype=np.float32)

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        cls = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:5])

        x1 = (xc - bw / 2) * img_w
        y1 = (yc - bh / 2) * img_h
        x2 = (xc + bw / 2) * img_w
        y2 = (yc + bh / 2) * img_h

        x1 = max(0, min(img_w - 1, x1))
        y1 = max(0, min(img_h - 1, y1))
        x2 = max(0, min(img_w - 1, x2))
        y2 = max(0, min(img_h - 1, y2))

        boxes.append([cls, x1, y1, x2, y2])

    return np.array(boxes, dtype=np.float32)

In [7]:
sequence_ids = sorted(list(sequences.keys()))

n_calib = max(1, int(len(sequence_ids) * CALIBRATION_SEQUENCE_RATIO))

calib_seq_ids = sequence_ids[:n_calib]
eval_seq_ids = sequence_ids[n_calib:]

if len(eval_seq_ids) == 0:
    eval_seq_ids = calib_seq_ids

print("Calibration sequences:", len(calib_seq_ids))
print("Evaluation sequences:", len(eval_seq_ids))

print("Calib examples:", calib_seq_ids[:5])
print("Eval examples:", eval_seq_ids[:5])

Calibration sequences: 8
Evaluation sequences: 32
Calib examples: ['MVI_39031', 'MVI_39051', 'MVI_39211', 'MVI_39271', 'MVI_39311']
Eval examples: ['MVI_39501', 'MVI_39511', 'MVI_40701', 'MVI_40711', 'MVI_40712']


In [9]:
def flatten_sequences(seq_ids):
    out = []
    for sid in seq_ids:
        for p in sequences[sid]:
            out.append((sid, p))
    return out


calib_stream = flatten_sequences(calib_seq_ids)
eval_stream = flatten_sequences(eval_seq_ids)

print("Calibration frames:", len(calib_stream))
print("Evaluation frames:", len(eval_stream))

Calibration frames: 11963
Evaluation frames: 44204


In [10]:
def read_gray_small(img_path, size=(160, 90)):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Image non lisible: {img_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, size)
    gray = gray.astype(np.float32) / 255.0

    return gray


def compute_visual_delta(img_path_t, img_path_prev):
    g_t = read_gray_small(img_path_t)
    g_prev = read_gray_small(img_path_prev)
    return float(np.mean(np.abs(g_t - g_prev)))


def compute_delta_cache(stream):
    """
    Delta par image.
    Si nouvelle séquence, delta = 1 pour forcer une inférence.
    """
    delta_cache = {}

    prev_sid = None
    prev_path = None

    for sid, img_path in tqdm(stream, desc="Computing visual deltas"):
        img_key = f"{sid}/{img_path.stem}"

        if prev_sid != sid or prev_path is None:
            delta_cache[img_key] = 1.0
        else:
            delta_cache[img_key] = compute_visual_delta(img_path, prev_path)

        prev_sid = sid
        prev_path = img_path

    return delta_cache


all_stream = flatten_sequences(sequence_ids)
delta_cache = compute_delta_cache(all_stream)

delta_values = np.array(list(delta_cache.values()))


Computing visual deltas: 100%|██████████| 56167/56167 [05:21<00:00, 174.77it/s]


In [11]:
MODEL_PATH = "/kaggle/input/models/chaymaerami/modelyolo/other/default/1/best (4).pt"
model = YOLO(MODEL_PATH)

In [12]:
def run_yolo_single(model, img_path):
    """
    Retourne:
    preds = [x1, y1, x2, y2, conf, cls]
    latency en secondes
    """
    start = time.perf_counter()

    results = model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRES,
        iou=IOU_NMS,
        verbose=False
    )

    latency = time.perf_counter() - start
    r = results[0]

    if r.boxes is None or len(r.boxes) == 0:
        return np.zeros((0, 6), dtype=np.float32), latency

    xyxy = r.boxes.xyxy.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy().reshape(-1, 1)
    cls = r.boxes.cls.cpu().numpy().reshape(-1, 1)

    preds = np.concatenate([xyxy, conf, cls], axis=1).astype(np.float32)
    return preds, latency


def make_key(sid, img_path):
    return f"{sid}/{Path(img_path).stem}"


def build_cache(stream):
    pred_cache = {}
    gt_cache = {}
    latency_cache = {}
    shape_cache = {}

    for sid, img_path in tqdm(stream, desc="YOLO cache"):
        key = make_key(sid, img_path)

        h, w = load_image_shape(img_path)
        label_path = find_label_path(img_path, IMG_VAL_DIR, LBL_VAL_DIR)

        gt = yolo_label_to_xyxy(label_path, w, h)
        preds, latency = run_yolo_single(model, img_path)

        pred_cache[key] = preds
        gt_cache[key] = gt
        latency_cache[key] = latency
        shape_cache[key] = (h, w)

    return pred_cache, gt_cache, latency_cache, shape_cache


pred_cache, gt_cache, latency_cache, shape_cache = build_cache(all_stream)

print("Mean YOLO latency:", np.mean(list(latency_cache.values())) * 1000, "ms")
print("Median YOLO latency:", np.median(list(latency_cache.values())) * 1000, "ms")

YOLO cache: 100%|██████████| 56167/56167 [20:08<00:00, 46.48it/s]

Mean YOLO latency: 18.204785635586724 ms
Median YOLO latency: 17.924858000014865 ms


# FrameHopper

In [13]:
# ================================================================
# FRAMEHOPPER - PAPER-BASED REIMPLEMENTATION FOR UA-DETRAC
# ================================================================
#
# Prerequisites already available:
#   all_stream
#   pred_cache
#   gt_cache
#   latency_cache
#   shape_cache
#   make_key(...)
#
# Required split:
#   calib_stream + eval_stream
# OR
#   CALIB_SIDS + EVAL_SIDS
#
# FrameHopper paper:
# - 3x3 pixel-change state representation
# - 10 states via MiniBatchKMeans
# - SARSA
# - action = number of skipped frames
# - D(fi,fj) = 1 - detection F1
# - IoU threshold = 0.5
# ================================================================

import os
import re
import cv2
import time
import math
import random
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict
from sklearn.cluster import MiniBatchKMeans


# ================================================================
# 0. CONFIGURATION
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ---------- FrameHopper state ----------
FH_GRID = 3
FH_NUM_STATES = 10
FH_KMEANS_BATCH = 30

# Lightweight state resolution.
# This is an implementation choice for this reimplementation.
FH_STATE_SIZE = (160, 90)      # (width, height)

# Pixel considered changed when absolute grayscale difference > threshold
# Paper does not explicitly provide this numerical threshold.
FH_PIXEL_CHANGE_THR = 20

# ---------- FrameHopper action space ----------
# UA-DETRAC reference stream = 25 FPS.
# FrameHopper paper permits max skip length ≈ one second of video.
FH_MAX_SKIP = 25

# ---------- FrameHopper target accuracy ----------
# Paper evaluates target F1 = 0.7, 0.8, 0.9.
# 0.80 is also used in their prototype experiment.
FH_TARGET_F1 = 0.80
FH_ERROR_THRESHOLD = 1.0 - FH_TARGET_F1   # theta = 0.20

# ---------- SARSA ----------
FH_EPISODES = 40
FH_ALPHA = 0.10
FH_GAMMA = 0.90

FH_EPS_START = 1.00
FH_EPS_END = 0.05
FH_EPS_DECAY = 0.92

# Reward:
#   + psi1 * (k + 1), if error <= theta
#   - psi2 * k,       otherwise
FH_PSI1 = 1.0
FH_PSI2 = 1.0

# F1 similarity matching
FH_MATCH_IOU = 0.50


# ================================================================
# 1. RESOLVE CALIBRATION / EVALUATION STREAMS
# ================================================================

def _resolve_split_stream(split):
    """
    Supports either:
        calib_stream / eval_stream
    or:
        calibration_stream / evaluation_stream
    or:
        CALIB_SIDS / EVAL_SIDS + all_stream
    """

    if split == "calib":
        candidates = [
            "calib_stream",
            "calibration_stream",
            "train_calib_stream"
        ]
        sid_candidates = [
            "CALIB_SIDS",
            "CALIB_SEQS",
            "CALIBRATION_SIDS"
        ]

    else:
        candidates = [
            "eval_stream",
            "evaluation_stream",
            "test_stream",
            "heldout_stream"
        ]
        sid_candidates = [
            "EVAL_SIDS",
            "EVAL_SEQS",
            "TEST_SIDS",
            "HELDOUT_SIDS"
        ]

    # Direct stream
    for name in candidates:
        if name in globals():
            stream = list(globals()[name])
            print(f"{split}: using variable `{name}` ({len(stream)} frames)")
            return stream

    # Filter all_stream by sequence IDs
    selected_sids = None

    for name in sid_candidates:
        if name in globals():
            selected_sids = set(globals()[name])
            print(f"{split}: using `{name}`")
            break

    if selected_sids is not None:
        stream = [
            (sid, path)
            for sid, path in all_stream
            if sid in selected_sids
        ]

        print(
            f"{split}: {len(selected_sids)} sequences, "
            f"{len(stream)} frames"
        )

        return stream

    raise RuntimeError(
        f"\nCannot identify {split} split.\n"
        f"Define `calib_stream` / `eval_stream`, or "
        f"`CALIB_SIDS` / `EVAL_SIDS` before running FrameHopper."
    )


fh_calib_stream = _resolve_split_stream("calib")
fh_eval_stream = _resolve_split_stream("eval")


# ================================================================
# 2. TEMPORAL ORDERING
# ================================================================

def extract_frame_number(img_path):
    """
    Extracts frame number from names such as:
    MVI_20011_img00001.jpg
    """
    stem = Path(img_path).stem

    m = re.search(r"_img(\d+)$", stem)

    if m:
        return int(m.group(1))

    nums = re.findall(r"\d+", stem)

    if not nums:
        raise ValueError(f"Cannot identify frame number: {img_path}")

    return int(nums[-1])


def group_stream_by_sequence(stream):

    grouped = defaultdict(list)

    for sid, img_path in stream:
        grouped[sid].append(
            (
                extract_frame_number(img_path),
                str(img_path)
            )
        )

    for sid in grouped:
        grouped[sid].sort(key=lambda z: z[0])

    return dict(grouped)


fh_calib_sequences = group_stream_by_sequence(fh_calib_stream)
fh_eval_sequences = group_stream_by_sequence(fh_eval_stream)

print("\nFrameHopper splits")
print("-------------------")
print("Calibration sequences:", len(fh_calib_sequences))
print("Calibration frames   :", sum(len(v) for v in fh_calib_sequences.values()))
print("Evaluation sequences :", len(fh_eval_sequences))
print("Evaluation frames    :", sum(len(v) for v in fh_eval_sequences.values()))


# ================================================================
# 3. IMAGE STATE REPRESENTATION
# ================================================================

def load_fh_gray(img_path):
    """
    Loads grayscale frame and resizes it for the lightweight
    FrameHopper state representation.
    """

    img = cv2.imread(str(img_path))

    if img is None:
        raise RuntimeError(f"Cannot read image: {img_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if FH_STATE_SIZE is not None:
        gray = cv2.resize(
            gray,
            FH_STATE_SIZE,
            interpolation=cv2.INTER_AREA
        )

    return gray


def fh_slice_feature(gray_prev, gray_curr):
    """
    FHop-Slice:
    divide frame into 3x3 regions and compute fraction
    of changed pixels in each region.

    Returns 9-dimensional feature vector.
    """

    if gray_prev.shape != gray_curr.shape:
        gray_curr = cv2.resize(
            gray_curr,
            (gray_prev.shape[1], gray_prev.shape[0])
        )

    diff = cv2.absdiff(gray_prev, gray_curr)

    changed = (diff > FH_PIXEL_CHANGE_THR).astype(np.float32)

    h, w = changed.shape

    features = []

    for gy in range(FH_GRID):

        y1 = gy * h // FH_GRID
        y2 = (gy + 1) * h // FH_GRID

        for gx in range(FH_GRID):

            x1 = gx * w // FH_GRID
            x2 = (gx + 1) * w // FH_GRID

            patch = changed[y1:y2, x1:x2]

            features.append(
                float(patch.mean()) if patch.size else 0.0
            )

    return np.asarray(features, dtype=np.float32)


# ================================================================
# 4. PRELOAD CALIBRATION FRAMES AT LOW RESOLUTION
# ================================================================

print("\nLoading calibration frames for FrameHopper states...")

fh_calib_gray = {}

for sid, frames in fh_calib_sequences.items():

    for frame_num, img_path in frames:

        key = make_key(sid, img_path)

        fh_calib_gray[key] = load_fh_gray(img_path)

print("Loaded calibration frames:", len(fh_calib_gray))


# ================================================================
# 5. BUILD FRAMEHOPPER STATE CLUSTERS
# ================================================================

print("\nBuilding 3x3 state features...")

state_features = []

for sid, frames in fh_calib_sequences.items():

    for i in range(1, len(frames)):

        _, prev_path = frames[i - 1]
        _, curr_path = frames[i]

        prev_key = make_key(sid, prev_path)
        curr_key = make_key(sid, curr_path)

        feat = fh_slice_feature(
            fh_calib_gray[prev_key],
            fh_calib_gray[curr_key]
        )

        state_features.append(feat)


state_features = np.asarray(
    state_features,
    dtype=np.float32
)

print("State feature matrix:", state_features.shape)


fh_kmeans = MiniBatchKMeans(
    n_clusters=FH_NUM_STATES,
    batch_size=FH_KMEANS_BATCH,
    random_state=SEED,
    n_init=10
)

fh_kmeans.fit(state_features)

print("MiniBatchKMeans fitted.")
print("Number of FrameHopper states:", FH_NUM_STATES)


# ================================================================
# 6. BOUNDING-BOX UTILITIES
# ================================================================

def _extract_pred_boxes(pred):
    """
    pred:
       [x1,y1,x2,y2,conf,cls]
    """
    pred = np.asarray(pred)

    if pred.size == 0:
        return np.zeros((0, 4), dtype=np.float32)

    return pred[:, :4].astype(np.float32)


def _extract_gt_boxes(gt):
    """
    Supports common formats:
       [x1,y1,x2,y2]
       [x1,y1,x2,y2,cls]
       [cls,x1,y1,x2,y2]
    """

    gt = np.asarray(gt)

    if gt.size == 0:
        return np.zeros((0, 4), dtype=np.float32)

    if gt.ndim == 1:
        gt = gt.reshape(1, -1)

    if gt.shape[1] == 4:
        return gt[:, :4].astype(np.float32)

    if gt.shape[1] >= 5:

        # likely [cls, x1,y1,x2,y2]
        first_col = gt[:, 0]

        if np.all(
            np.isclose(first_col, np.round(first_col))
        ) and np.nanmax(np.abs(first_col)) <= 10:

            return gt[:, 1:5].astype(np.float32)

        # likely [x1,y1,x2,y2,cls]
        return gt[:, :4].astype(np.float32)

    raise ValueError(f"Unsupported GT format: {gt.shape}")


def box_iou_matrix(boxes_a, boxes_b):

    boxes_a = np.asarray(boxes_a, dtype=np.float32)
    boxes_b = np.asarray(boxes_b, dtype=np.float32)

    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros(
            (len(boxes_a), len(boxes_b)),
            dtype=np.float32
        )

    ax1 = boxes_a[:, 0][:, None]
    ay1 = boxes_a[:, 1][:, None]
    ax2 = boxes_a[:, 2][:, None]
    ay2 = boxes_a[:, 3][:, None]

    bx1 = boxes_b[:, 0][None, :]
    by1 = boxes_b[:, 1][None, :]
    bx2 = boxes_b[:, 2][None, :]
    by2 = boxes_b[:, 3][None, :]

    inter_x1 = np.maximum(ax1, bx1)
    inter_y1 = np.maximum(ay1, by1)

    inter_x2 = np.minimum(ax2, bx2)
    inter_y2 = np.minimum(ay2, by2)

    inter_w = np.maximum(0.0, inter_x2 - inter_x1)
    inter_h = np.maximum(0.0, inter_y2 - inter_y1)

    inter = inter_w * inter_h

    area_a = np.maximum(
        0.0,
        (ax2 - ax1) * (ay2 - ay1)
    )

    area_b = np.maximum(
        0.0,
        (bx2 - bx1) * (by2 - by1)
    )

    union = area_a + area_b - inter

    return inter / np.maximum(union, 1e-9)


# ================================================================
# 7. DETECTION F1 BETWEEN TWO CONTINUOUS YOLO OUTPUTS
# ================================================================

def detection_f1_between_predictions(
    pred_a,
    pred_b,
    iou_thr=FH_MATCH_IOU
):
    """
    FrameHopper similarity metric.

    D(fi,fj) = 1 - F1(fi,fj)

    Both empty => F1 = 1.
    """

    boxes_a = _extract_pred_boxes(pred_a)
    boxes_b = _extract_pred_boxes(pred_b)

    na = len(boxes_a)
    nb = len(boxes_b)

    if na == 0 and nb == 0:
        return 1.0

    if na == 0 or nb == 0:
        return 0.0

    ious = box_iou_matrix(
        boxes_a,
        boxes_b
    )

    hits = 0

    used_a = set()
    used_b = set()

    # Greedy highest-IoU one-to-one matching
    flat_order = np.argsort(
        ious.ravel()
    )[::-1]

    for flat_idx in flat_order:

        ia, ib = np.unravel_index(
            flat_idx,
            ious.shape
        )

        if ious[ia, ib] < iou_thr:
            break

        if ia in used_a or ib in used_b:
            continue

        used_a.add(ia)
        used_b.add(ib)

        hits += 1

    precision = hits / na
    recall = hits / nb

    if precision + recall == 0:
        return 0.0

    return (
        2.0 * precision * recall
        / (precision + recall)
    )


def fh_detection_distance(pred_a, pred_b):

    return (
        1.0
        - detection_f1_between_predictions(
            pred_a,
            pred_b,
            FH_MATCH_IOU
        )
    )


# ================================================================
# 8. FRAMEHOPPER STATE FUNCTION
# ================================================================

def fh_state_from_gray(gray_prev, gray_curr):

    feat = fh_slice_feature(
        gray_prev,
        gray_curr
    )

    return int(
        fh_kmeans.predict(
            feat.reshape(1, -1)
        )[0]
    )


# ================================================================
# 9. SARSA AGENT
# ================================================================

FH_NUM_ACTIONS = FH_MAX_SKIP + 1

# action k = number of skipped frames
# k = 0 => next frame also processed

Q = np.zeros(
    (FH_NUM_STATES, FH_NUM_ACTIONS),
    dtype=np.float32
)


def epsilon_greedy(Q, state, epsilon):

    if np.random.rand() < epsilon:
        return np.random.randint(FH_NUM_ACTIONS)

    values = Q[state]

    max_value = np.max(values)

    candidates = np.flatnonzero(
        np.isclose(values, max_value)
    )

    return int(
        np.random.choice(candidates)
    )


def fh_reward(error, actual_skip):

    if error <= FH_ERROR_THRESHOLD:

        return (
            FH_PSI1
            * (actual_skip + 1)
        )

    return (
        -FH_PSI2
        * actual_skip
    )


# ================================================================
# 10. SARSA TRAINING
# ================================================================

print("\nTraining FrameHopper SARSA agent...")
print("Target F1:", FH_TARGET_F1)
print("Error threshold theta:", FH_ERROR_THRESHOLD)
print("Maximum skip:", FH_MAX_SKIP)

epsilon = FH_EPS_START

training_history = []

sequence_ids = list(
    fh_calib_sequences.keys()
)


for episode in range(FH_EPISODES):

    random.shuffle(sequence_ids)

    ep_reward = 0.0
    ep_actions = 0

    for sid in sequence_ids:

        frames = fh_calib_sequences[sid]

        n = len(frames)

        if n < 3:
            continue

        # First two frames are processed to establish temporal state
        prev_idx = 0
        curr_idx = 1

        _, prev_path = frames[prev_idx]
        _, curr_path = frames[curr_idx]

        prev_key = make_key(sid, prev_path)
        curr_key = make_key(sid, curr_path)

        state = fh_state_from_gray(
            fh_calib_gray[prev_key],
            fh_calib_gray[curr_key]
        )

        action = epsilon_greedy(
            Q,
            state,
            epsilon
        )

        while curr_idx < n - 1:

            requested_skip = int(action)

            next_idx = min(
                curr_idx + requested_skip + 1,
                n - 1
            )

            actual_skip = max(
                0,
                next_idx - curr_idx - 1
            )

            _, next_path = frames[next_idx]

            current_key = make_key(
                sid,
                frames[curr_idx][1]
            )

            next_key = make_key(
                sid,
                next_path
            )

            # FrameHopper detection-output distance
            error = fh_detection_distance(
                pred_cache[current_key],
                pred_cache[next_key]
            )

            reward = fh_reward(
                error,
                actual_skip
            )

            ep_reward += reward
            ep_actions += 1

            terminal = (
                next_idx >= n - 1
            )

            if terminal:

                td_target = reward

                Q[state, action] += (
                    FH_ALPHA
                    * (
                        td_target
                        - Q[state, action]
                    )
                )

                break

            # New processed frame
            _, old_path = frames[curr_idx]
            _, new_path = frames[next_idx]

            old_key = make_key(
                sid,
                old_path
            )

            new_key = make_key(
                sid,
                new_path
            )

            new_state = fh_state_from_gray(
                fh_calib_gray[old_key],
                fh_calib_gray[new_key]
            )

            new_action = epsilon_greedy(
                Q,
                new_state,
                epsilon
            )

            # SARSA update
            td_target = (
                reward
                + FH_GAMMA
                * Q[new_state, new_action]
            )

            Q[state, action] += (
                FH_ALPHA
                * (
                    td_target
                    - Q[state, action]
                )
            )

            prev_idx = curr_idx
            curr_idx = next_idx

            state = new_state
            action = new_action

    epsilon = max(
        FH_EPS_END,
        epsilon * FH_EPS_DECAY
    )

    mean_reward = (
        ep_reward / max(ep_actions, 1)
    )

    training_history.append(
        {
            "episode": episode + 1,
            "epsilon": epsilon,
            "mean_reward": mean_reward
        }
    )

    if (
        episode == 0
        or (episode + 1) % 5 == 0
        or episode == FH_EPISODES - 1
    ):

        print(
            f"Episode {episode+1:02d}/{FH_EPISODES} | "
            f"epsilon={epsilon:.3f} | "
            f"mean reward={mean_reward:.4f}"
        )


print("\nSARSA training completed.")


# ================================================================
# 11. FINAL DETERMINISTIC POLICY
# ================================================================

fh_policy = np.argmax(
    Q,
    axis=1
).astype(int)

print("\nLearned FrameHopper policy")
print("--------------------------")

for s, k in enumerate(fh_policy):

    print(
        f"State {s:02d} -> skip {k:02d} frame(s)"
    )


# ================================================================
# 12. RUN FRAMEHOPPER ON HELD-OUT EVALUATION
# ================================================================

def run_framehopper_on_sequences(
    sequences,
    policy,
    verbose=True
):

    output_cache = {}

    executed_cache = {}
    reuse_age_cache = {}

    schedule_rows = []

    scheduler_time_total = 0.0
    detector_time_total = 0.0

    total_frames = 0
    total_detector_calls = 0

    for sid, frames in sequences.items():

        n = len(frames)

        total_frames += n

        if n == 0:
            continue

        if verbose:
            print(
                f"FrameHopper evaluation: "
                f"{sid} ({n} frames)"
            )

        # --------------------------------------------------------
        # FIRST FRAME -> detector execution
        # --------------------------------------------------------

        _, path0 = frames[0]
        key0 = make_key(sid, path0)

        output_cache[key0] = (
            pred_cache[key0].copy()
        )

        executed_cache[key0] = True
        reuse_age_cache[key0] = 0

        detector_time_total += (
            latency_cache[key0]
        )

        total_detector_calls += 1

        schedule_rows.append(
            {
                "sequence": sid,
                "frame": frames[0][0],
                "key": key0,
                "executed": 1,
                "reuse_age": 0,
                "skip_action": 0,
                "state": -1
            }
        )

        if n == 1:
            continue

        # --------------------------------------------------------
        # SECOND FRAME -> detector execution to establish state
        # --------------------------------------------------------

        curr_idx = 1
        prev_processed_idx = 0

        _, path1 = frames[curr_idx]
        key1 = make_key(sid, path1)

        output_cache[key1] = (
            pred_cache[key1].copy()
        )

        executed_cache[key1] = True
        reuse_age_cache[key1] = 0

        detector_time_total += (
            latency_cache[key1]
        )

        total_detector_calls += 1

        prev_gray = load_fh_gray(
            frames[prev_processed_idx][1]
        )

        curr_gray = load_fh_gray(
            frames[curr_idx][1]
        )

        # --------------------------------------------------------
        # ADAPTIVE SKIPPING
        # --------------------------------------------------------

        while True:

            # Scheduler cost only:
            # images already considered available in memory
            t_sched = time.perf_counter()

            state = fh_state_from_gray(
                prev_gray,
                curr_gray
            )

            skip_action = int(
                policy[state]
            )

            scheduler_time_total += (
                time.perf_counter()
                - t_sched
            )

            # Current frame record
            current_key = make_key(
                sid,
                frames[curr_idx][1]
            )

            # avoid duplicate row only for frame 1
            if curr_idx == 1:

                schedule_rows.append(
                    {
                        "sequence": sid,
                        "frame": frames[curr_idx][0],
                        "key": current_key,
                        "executed": 1,
                        "reuse_age": 0,
                        "skip_action": skip_action,
                        "state": state
                    }
                )

            next_idx = (
                curr_idx
                + skip_action
                + 1
            )

            # ----------------------------------------------------
            # If skip reaches end of sequence:
            # reuse current detector output for remaining frames
            # ----------------------------------------------------

            if next_idx >= n:

                age = 0

                for j in range(
                    curr_idx + 1,
                    n
                ):

                    age += 1

                    _, skipped_path = frames[j]

                    skipped_key = make_key(
                        sid,
                        skipped_path
                    )

                    output_cache[skipped_key] = (
                        pred_cache[current_key].copy()
                    )

                    executed_cache[
                        skipped_key
                    ] = False

                    reuse_age_cache[
                        skipped_key
                    ] = age

                    schedule_rows.append(
                        {
                            "sequence": sid,
                            "frame": frames[j][0],
                            "key": skipped_key,
                            "executed": 0,
                            "reuse_age": age,
                            "skip_action": skip_action,
                            "state": state
                        }
                    )

                break

            # ----------------------------------------------------
            # Reuse detections on skipped frames
            # ----------------------------------------------------

            age = 0

            for j in range(
                curr_idx + 1,
                next_idx
            ):

                age += 1

                _, skipped_path = frames[j]

                skipped_key = make_key(
                    sid,
                    skipped_path
                )

                output_cache[skipped_key] = (
                    pred_cache[current_key].copy()
                )

                executed_cache[
                    skipped_key
                ] = False

                reuse_age_cache[
                    skipped_key
                ] = age

                schedule_rows.append(
                    {
                        "sequence": sid,
                        "frame": frames[j][0],
                        "key": skipped_key,
                        "executed": 0,
                        "reuse_age": age,
                        "skip_action": skip_action,
                        "state": state
                    }
                )

            # ----------------------------------------------------
            # NEXT PROCESSED FRAME -> detector
            # ----------------------------------------------------

            prev_processed_idx = curr_idx
            curr_idx = next_idx

            _, next_path = frames[curr_idx]

            next_key = make_key(
                sid,
                next_path
            )

            output_cache[next_key] = (
                pred_cache[next_key].copy()
            )

            executed_cache[
                next_key
            ] = True

            reuse_age_cache[
                next_key
            ] = 0

            detector_time_total += (
                latency_cache[next_key]
            )

            total_detector_calls += 1

            # Update grayscale references
            prev_gray = curr_gray
            curr_gray = load_fh_gray(
                next_path
            )

            # Next iteration records detector frame
            t_sched = time.perf_counter()

            next_state = fh_state_from_gray(
                prev_gray,
                curr_gray
            )

            next_skip = int(
                policy[next_state]
            )

            scheduler_time_total += (
                time.perf_counter()
                - t_sched
            )

            schedule_rows.append(
                {
                    "sequence": sid,
                    "frame": frames[curr_idx][0],
                    "key": next_key,
                    "executed": 1,
                    "reuse_age": 0,
                    "skip_action": next_skip,
                    "state": next_state
                }
            )

            # If last frame
            if curr_idx >= n - 1:
                break

    return {
        "output_cache": output_cache,
        "executed_cache": executed_cache,
        "reuse_age_cache": reuse_age_cache,
        "schedule": pd.DataFrame(
            schedule_rows
        ),
        "total_frames": total_frames,
        "detector_calls": total_detector_calls,
        "detector_time": detector_time_total,
        "scheduler_time": scheduler_time_total
    }


fh_eval = run_framehopper_on_sequences(
    fh_eval_sequences,
    fh_policy,
    verbose=True
)


fh_output_cache = fh_eval[
    "output_cache"
]


# ================================================================
# 13. FIXED-IoU PRECISION / RECALL / F1 / COUNT MAE
# ================================================================

def match_pred_gt(
    pred,
    gt,
    iou_thr=0.50
):

    pred_boxes = _extract_pred_boxes(
        pred
    )

    gt_boxes = _extract_gt_boxes(
        gt
    )

    npred = len(pred_boxes)
    ngt = len(gt_boxes)

    if npred == 0 and ngt == 0:
        return 0, 0, 0

    if npred == 0:
        return 0, 0, ngt

    if ngt == 0:
        return 0, npred, 0

    ious = box_iou_matrix(
        pred_boxes,
        gt_boxes
    )

    tp = 0

    used_p = set()
    used_g = set()

    flat_order = np.argsort(
        ious.ravel()
    )[::-1]

    for idx in flat_order:

        ip, ig = np.unravel_index(
            idx,
            ious.shape
        )

        if ious[ip, ig] < iou_thr:
            break

        if (
            ip in used_p
            or ig in used_g
        ):
            continue

        used_p.add(ip)
        used_g.add(ig)

        tp += 1

    fp = npred - tp
    fn = ngt - tp

    return tp, fp, fn


# ordered evaluation keys
fh_eval_keys = []

for sid, frames in fh_eval_sequences.items():

    for _, img_path in frames:

        fh_eval_keys.append(
            make_key(sid, img_path)
        )


TP = 0
FP = 0
FN = 0

count_errors = []

for key in fh_eval_keys:

    pred = fh_output_cache[key]
    gt = gt_cache[key]

    tp, fp, fn = match_pred_gt(
        pred,
        gt,
        iou_thr=0.50
    )

    TP += tp
    FP += fp
    FN += fn

    n_pred = len(
        _extract_pred_boxes(pred)
    )

    n_gt = len(
        _extract_gt_boxes(gt)
    )

    count_errors.append(
        abs(n_pred - n_gt)
    )


precision = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else 0.0
)

recall = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0.0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

count_mae = float(
    np.mean(count_errors)
)


# ================================================================
# 14. SINGLE-CLASS AP
# ================================================================

def compute_ap_at_iou(
    output_cache,
    gt_cache,
    keys,
    iou_thr
):

    detections = []

    total_gt = 0

    gt_boxes_by_key = {}

    for key in keys:

        gt_boxes = _extract_gt_boxes(
            gt_cache[key]
        )

        gt_boxes_by_key[key] = gt_boxes

        total_gt += len(gt_boxes)

        preds = np.asarray(
            output_cache[key]
        )

        if preds.size == 0:
            continue

        if preds.ndim == 1:
            preds = preds.reshape(1, -1)

        for p in preds:

            conf = (
                float(p[4])
                if len(p) > 4
                else 1.0
            )

            detections.append(
                (
                    conf,
                    key,
                    p[:4].astype(
                        np.float32
                    )
                )
            )

    if total_gt == 0:
        return 0.0

    detections.sort(
        key=lambda x: x[0],
        reverse=True
    )

    matched = {
        key: np.zeros(
            len(gt_boxes_by_key[key]),
            dtype=bool
        )
        for key in keys
    }

    tp_arr = []
    fp_arr = []

    for conf, key, pbox in detections:

        gt_boxes = gt_boxes_by_key[key]

        if len(gt_boxes) == 0:

            tp_arr.append(0)
            fp_arr.append(1)
            continue

        ious = box_iou_matrix(
            pbox.reshape(1, 4),
            gt_boxes
        )[0]

        order = np.argsort(
            ious
        )[::-1]

        found = False

        for gi in order:

            if ious[gi] < iou_thr:
                break

            if not matched[key][gi]:

                matched[key][gi] = True
                found = True
                break

        if found:

            tp_arr.append(1)
            fp_arr.append(0)

        else:

            tp_arr.append(0)
            fp_arr.append(1)

    if len(tp_arr) == 0:
        return 0.0

    tp_cum = np.cumsum(
        np.asarray(tp_arr)
    )

    fp_cum = np.cumsum(
        np.asarray(fp_arr)
    )

    recalls = (
        tp_cum / total_gt
    )

    precisions = (
        tp_cum
        / np.maximum(
            tp_cum + fp_cum,
            1e-12
        )
    )

    # COCO-like 101-point interpolation
    recall_grid = np.linspace(
        0.0,
        1.0,
        101
    )

    interp_precision = []

    for r in recall_grid:

        mask = recalls >= r

        if np.any(mask):
            interp_precision.append(
                np.max(
                    precisions[mask]
                )
            )
        else:
            interp_precision.append(
                0.0
            )

    return float(
        np.mean(
            interp_precision
        )
    )


print("\nCalculating FrameHopper AP...")

fh_ap50 = compute_ap_at_iou(
    fh_output_cache,
    gt_cache,
    fh_eval_keys,
    0.50
)

fh_iou_thresholds = np.arange(
    0.50,
    0.96,
    0.05
)

fh_aps = []

for thr in fh_iou_thresholds:

    ap = compute_ap_at_iou(
        fh_output_cache,
        gt_cache,
        fh_eval_keys,
        float(thr)
    )

    fh_aps.append(ap)

fh_map5095 = float(
    np.mean(fh_aps)
)


# ================================================================
# 15. COMPUTATIONAL METRICS
# ================================================================

n_total = fh_eval[
    "total_frames"
]

n_detector = fh_eval[
    "detector_calls"
]

n_reused = (
    n_total
    - n_detector
)

frame_reduction = (
    1.0
    - n_detector / n_total
)

detector_runtime = fh_eval[
    "detector_time"
]

scheduler_runtime = fh_eval[
    "scheduler_time"
]

total_runtime = (
    detector_runtime
    + scheduler_runtime
)

mean_latency_ms = (
    1000.0
    * total_runtime
    / n_total
)

processing_rate = (
    n_total
    / total_runtime
    if total_runtime > 0
    else 0.0
)


# ================================================================
# 16. FINAL FRAMEHOPPER RESULT
# ================================================================

framehopper_results = pd.DataFrame(
    [
        {
            "Method": "FrameHopper",
            "Target F1": FH_TARGET_F1,
            "mAP@0.5 (%)": fh_ap50 * 100,
            "mAP@0.5:0.95 (%)": fh_map5095 * 100,
            "Precision (%)": precision * 100,
            "Recall (%)": recall * 100,
            "F1 (%)": f1 * 100,
            "Count MAE": count_mae,
            "Total Frames": n_total,
            "Detector Executions": n_detector,
            "Reused Frames": n_reused,
            "Frame Reduction (%)": frame_reduction * 100,
            "Detector Runtime (s)": detector_runtime,
            "Scheduler Runtime (s)": scheduler_runtime,
            "Mean Latency (ms)": mean_latency_ms,
            "Proc. Rate (FPS)": processing_rate
        }
    ]
)


print("\n")
print("=" * 72)
print("FRAMEHOPPER FINAL RESULTS")
print("=" * 72)

print(
    framehopper_results
    .T
    .to_string(
        header=False
    )
)


# ================================================================
# 17. SAVE ARTIFACTS
# ================================================================

np.save(
    "framehopper_Q.npy",
    Q
)

joblib.dump(
    fh_kmeans,
    "framehopper_kmeans.pkl"
)

fh_eval[
    "schedule"
].to_csv(
    "framehopper_eval_schedule.csv",
    index=False
)

framehopper_results.to_csv(
    "framehopper_results.csv",
    index=False
)

pd.DataFrame(
    training_history
).to_csv(
    "framehopper_training_history.csv",
    index=False
)

print("\nSaved:")
print(" - framehopper_Q.npy")
print(" - framehopper_kmeans.pkl")
print(" - framehopper_eval_schedule.csv")
print(" - framehopper_results.csv")
print(" - framehopper_training_history.csv")

calib: using variable `calib_stream` (11963 frames)
eval: using variable `eval_stream` (44204 frames)

FrameHopper splits
-------------------
Calibration sequences: 8
Calibration frames   : 11963
Evaluation sequences : 32
Evaluation frames    : 44204

Loading calibration frames for FrameHopper states...
Loaded calibration frames: 11963

Building 3x3 state features...
State feature matrix: (11955, 9)
MiniBatchKMeans fitted.
Number of FrameHopper states: 10

Training FrameHopper SARSA agent...
Target F1: 0.8
Error threshold theta: 0.19999999999999996
Maximum skip: 25
Episode 01/40 | epsilon=0.920 | mean reward=-9.7576
Episode 05/40 | epsilon=0.659 | mean reward=-6.7360
Episode 10/40 | epsilon=0.434 | mean reward=-3.7289
Episode 15/40 | epsilon=0.286 | mean reward=-1.6447
Episode 20/40 | epsilon=0.189 | mean reward=-0.2136
Episode 25/40 | epsilon=0.124 | mean reward=0.6749
Episode 30/40 | epsilon=0.082 | mean reward=1.0287
Episode 35/40 | epsilon=0.054 | mean reward=1.3996
Episode 40/40 |

In [14]:
# ================================================================
# OPTICAL-FLOW-BASED PROPAGATION BASELINE
# ================================================================
#
# Principle:
#   - YOLO11s is executed every P_FS = 3 frames
#   - Intermediate detections are propagated using dense
#     Farneback optical flow
#   - Same temporally ordered held-out evaluation sequences
#   - Same YOLO cache and detector settings
#
# Requirements already available:
#   pred_cache
#   gt_cache
#   latency_cache
#   shape_cache
#   all_stream
#   make_key(...)
#
# Evaluation functions from the FrameHopper cell:
#   _extract_pred_boxes(...)
#   _extract_gt_boxes(...)
#   match_pred_gt(...)
#   compute_ap_at_iou(...)
# ================================================================

import os
import re
import cv2
import time
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict


# ================================================================
# 0. CONFIGURATION
# ================================================================

# Detector-to-detector period.
# P_FS = 3 means:
# DETECT -> PROPAGATE -> PROPAGATE -> DETECT
OF_PERIOD = 3

# Resolution used only for optical-flow calculation.
# Flow is rescaled back to original image coordinates.
OF_WIDTH = 320
OF_HEIGHT = 180

# Farneback parameters
OF_PYR_SCALE = 0.5
OF_LEVELS = 3
OF_WINSIZE = 15
OF_ITERATIONS = 3
OF_POLY_N = 5
OF_POLY_SIGMA = 1.2
OF_FLAGS = 0

# Robust motion estimation inside each bounding box
OF_USE_MEDIAN = True


# ================================================================
# 1. GET HELD-OUT EVALUATION SEQUENCES
# ================================================================

def extract_frame_number_of(img_path):

    stem = Path(img_path).stem

    m = re.search(r"_img(\d+)$", stem)

    if m:
        return int(m.group(1))

    nums = re.findall(r"\d+", stem)

    if not nums:
        raise ValueError(
            f"Cannot identify frame number: {img_path}"
        )

    return int(nums[-1])


def group_stream_by_sequence_of(stream):

    grouped = defaultdict(list)

    for sid, img_path in stream:

        grouped[sid].append(
            (
                extract_frame_number_of(img_path),
                str(img_path)
            )
        )

    for sid in grouped:

        grouped[sid].sort(
            key=lambda z: z[0]
        )

    return dict(grouped)


# Reuse FrameHopper held-out sequences if available
if "fh_eval_sequences" in globals():

    of_eval_sequences = fh_eval_sequences

    print(
        "Using the same held-out evaluation "
        "sequences as FrameHopper."
    )

elif "eval_stream" in globals():

    of_eval_sequences = group_stream_by_sequence_of(
        eval_stream
    )

elif "EVAL_SIDS" in globals():

    eval_sid_set = set(EVAL_SIDS)

    tmp_stream = [
        (sid, img_path)
        for sid, img_path in all_stream
        if sid in eval_sid_set
    ]

    of_eval_sequences = group_stream_by_sequence_of(
        tmp_stream
    )

elif "EVAL_SEQS" in globals():

    eval_sid_set = set(EVAL_SEQS)

    tmp_stream = [
        (sid, img_path)
        for sid, img_path in all_stream
        if sid in eval_sid_set
    ]

    of_eval_sequences = group_stream_by_sequence_of(
        tmp_stream
    )

else:

    raise RuntimeError(
        "Cannot identify held-out evaluation split. "
        "Use the same eval split as RAED-VP."
    )


print(
    "Evaluation sequences:",
    len(of_eval_sequences)
)

print(
    "Evaluation frames:",
    sum(
        len(v)
        for v in of_eval_sequences.values()
    )
)


# ================================================================
# 2. IMAGE LOADING FOR OPTICAL FLOW
# ================================================================

def load_flow_gray(img_path):

    img = cv2.imread(
        str(img_path),
        cv2.IMREAD_GRAYSCALE
    )

    if img is None:

        raise RuntimeError(
            f"Cannot read image: {img_path}"
        )

    original_h, original_w = img.shape[:2]

    small = cv2.resize(
        img,
        (OF_WIDTH, OF_HEIGHT),
        interpolation=cv2.INTER_AREA
    )

    return (
        small,
        original_h,
        original_w
    )


# ================================================================
# 3. DENSE FARNEBACK FLOW
# ================================================================

def compute_dense_flow(
    gray_prev,
    gray_curr
):

    flow = cv2.calcOpticalFlowFarneback(
        gray_prev,
        gray_curr,
        None,
        OF_PYR_SCALE,
        OF_LEVELS,
        OF_WINSIZE,
        OF_ITERATIONS,
        OF_POLY_N,
        OF_POLY_SIGMA,
        OF_FLAGS
    )

    return flow


# ================================================================
# 4. PROPAGATE ONE SET OF DETECTIONS
# ================================================================

def propagate_boxes_with_flow(
    detections,
    flow,
    original_h,
    original_w
):
    """
    detections:
        [x1, y1, x2, y2, conf, cls]

    flow:
        optical flow at OF_WIDTH x OF_HEIGHT

    Bounding-box displacement is estimated from
    the median optical-flow vector inside each box.
    """

    detections = np.asarray(
        detections,
        dtype=np.float32
    )

    if detections.size == 0:

        return np.zeros(
            (0, 6),
            dtype=np.float32
        )

    if detections.ndim == 1:

        detections = detections.reshape(
            1, -1
        )

    propagated = detections.copy()

    scale_x = (
        OF_WIDTH / original_w
    )

    scale_y = (
        OF_HEIGHT / original_h
    )

    inv_scale_x = (
        original_w / OF_WIDTH
    )

    inv_scale_y = (
        original_h / OF_HEIGHT
    )

    for i, det in enumerate(detections):

        x1, y1, x2, y2 = det[:4]

        # Convert bbox to flow resolution
        sx1 = int(
            np.clip(
                round(x1 * scale_x),
                0,
                OF_WIDTH - 1
            )
        )

        sy1 = int(
            np.clip(
                round(y1 * scale_y),
                0,
                OF_HEIGHT - 1
            )
        )

        sx2 = int(
            np.clip(
                round(x2 * scale_x),
                sx1 + 1,
                OF_WIDTH
            )
        )

        sy2 = int(
            np.clip(
                round(y2 * scale_y),
                sy1 + 1,
                OF_HEIGHT
            )
        )

        roi_flow = flow[
            sy1:sy2,
            sx1:sx2
        ]

        if roi_flow.size == 0:
            continue

        fx = roi_flow[..., 0]
        fy = roi_flow[..., 1]

        # Remove invalid values
        valid = (
            np.isfinite(fx)
            & np.isfinite(fy)
        )

        if not np.any(valid):
            continue

        if OF_USE_MEDIAN:

            dx_small = float(
                np.median(fx[valid])
            )

            dy_small = float(
                np.median(fy[valid])
            )

        else:

            dx_small = float(
                np.mean(fx[valid])
            )

            dy_small = float(
                np.mean(fy[valid])
            )

        # Convert displacement to original resolution
        dx = (
            dx_small
            * inv_scale_x
        )

        dy = (
            dy_small
            * inv_scale_y
        )

        nx1 = x1 + dx
        ny1 = y1 + dy
        nx2 = x2 + dx
        ny2 = y2 + dy

        # Clip to image
        nx1 = np.clip(
            nx1,
            0,
            original_w - 1
        )

        ny1 = np.clip(
            ny1,
            0,
            original_h - 1
        )

        nx2 = np.clip(
            nx2,
            nx1 + 1,
            original_w
        )

        ny2 = np.clip(
            ny2,
            ny1 + 1,
            original_h
        )

        propagated[i, :4] = [
            nx1,
            ny1,
            nx2,
            ny2
        ]

    return propagated.astype(
        np.float32
    )


# ================================================================
# 5. RUN OPTICAL-FLOW PROPAGATION
# ================================================================

def run_optical_flow_baseline(
    sequences,
    period=OF_PERIOD,
    verbose=True
):

    output_cache = {}
    executed_cache = {}

    rows = []

    detector_runtime = 0.0
    propagation_runtime = 0.0

    total_frames = 0
    detector_calls = 0

    for sid, frames in sequences.items():

        n = len(frames)

        total_frames += n

        if verbose:

            print(
                f"Optical-flow baseline: "
                f"{sid} ({n} frames)"
            )

        if n == 0:
            continue

        previous_gray = None
        current_predictions = None

        for idx, (
            frame_num,
            img_path
        ) in enumerate(frames):

            key = make_key(
                sid,
                img_path
            )

            # ====================================================
            # DETECTOR FRAME
            # ====================================================

            if idx % period == 0:

                current_predictions = (
                    pred_cache[key].copy()
                )

                output_cache[key] = (
                    current_predictions.copy()
                )

                executed_cache[key] = True

                detector_runtime += (
                    latency_cache[key]
                )

                detector_calls += 1

                # Load current gray frame as reference
                previous_gray, h, w = (
                    load_flow_gray(
                        img_path
                    )
                )

                rows.append(
                    {
                        "sequence": sid,
                        "frame": frame_num,
                        "key": key,
                        "detector": 1,
                        "propagated": 0
                    }
                )

                continue

            # ====================================================
            # PROPAGATED FRAME
            # ====================================================

            t0 = time.perf_counter()

            current_gray, h, w = (
                load_flow_gray(
                    img_path
                )
            )

            flow = compute_dense_flow(
                previous_gray,
                current_gray
            )

            current_predictions = (
                propagate_boxes_with_flow(
                    current_predictions,
                    flow,
                    h,
                    w
                )
            )

            propagation_runtime += (
                time.perf_counter()
                - t0
            )

            output_cache[key] = (
                current_predictions.copy()
            )

            executed_cache[key] = False

            previous_gray = current_gray

            rows.append(
                {
                    "sequence": sid,
                    "frame": frame_num,
                    "key": key,
                    "detector": 0,
                    "propagated": 1
                }
            )

    return {
        "output_cache": output_cache,
        "executed_cache": executed_cache,
        "schedule": pd.DataFrame(rows),
        "total_frames": total_frames,
        "detector_calls": detector_calls,
        "detector_runtime": detector_runtime,
        "propagation_runtime": propagation_runtime
    }


print("\nRunning optical-flow propagation...\n")

of_eval = run_optical_flow_baseline(
    of_eval_sequences,
    period=OF_PERIOD,
    verbose=True
)

of_output_cache = (
    of_eval["output_cache"]
)


# ================================================================
# 6. ORDERED EVALUATION KEYS
# ================================================================

of_eval_keys = []

for sid, frames in of_eval_sequences.items():

    for _, img_path in frames:

        of_eval_keys.append(
            make_key(
                sid,
                img_path
            )
        )


# ================================================================
# 7. PRECISION / RECALL / F1 / COUNT MAE
# ================================================================

TP = 0
FP = 0
FN = 0

count_errors = []

for key in of_eval_keys:

    pred = of_output_cache[key]
    gt = gt_cache[key]

    tp, fp, fn = match_pred_gt(
        pred,
        gt,
        iou_thr=0.50
    )

    TP += tp
    FP += fp
    FN += fn

    n_pred = len(
        _extract_pred_boxes(pred)
    )

    n_gt = len(
        _extract_gt_boxes(gt)
    )

    count_errors.append(
        abs(n_pred - n_gt)
    )


precision_of = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else 0.0
)

recall_of = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0.0
)

f1_of = (
    2
    * precision_of
    * recall_of
    / (
        precision_of
        + recall_of
    )
    if (
        precision_of
        + recall_of
    ) > 0
    else 0.0
)

count_mae_of = float(
    np.mean(
        count_errors
    )
)


# ================================================================
# 8. mAP@0.5
# ================================================================

print(
    "\nCalculating optical-flow AP..."
)

of_ap50 = compute_ap_at_iou(
    of_output_cache,
    gt_cache,
    of_eval_keys,
    0.50
)


# ================================================================
# 9. mAP@0.5:0.95
# ================================================================

of_aps = []

for thr in np.arange(
    0.50,
    0.96,
    0.05
):

    ap = compute_ap_at_iou(
        of_output_cache,
        gt_cache,
        of_eval_keys,
        float(thr)
    )

    of_aps.append(ap)


of_map5095 = float(
    np.mean(of_aps)
)


# ================================================================
# 10. COMPUTATIONAL METRICS
# ================================================================

n_total_of = (
    of_eval["total_frames"]
)

n_detector_of = (
    of_eval["detector_calls"]
)

n_propagated_of = (
    n_total_of
    - n_detector_of
)

frame_reduction_of = (
    1.0
    - n_detector_of
    / n_total_of
)

detector_runtime_of = (
    of_eval[
        "detector_runtime"
    ]
)

propagation_runtime_of = (
    of_eval[
        "propagation_runtime"
    ]
)

total_runtime_of = (
    detector_runtime_of
    + propagation_runtime_of
)

mean_latency_of = (
    1000.0
    * total_runtime_of
    / n_total_of
)

processing_rate_of = (
    n_total_of
    / total_runtime_of
    if total_runtime_of > 0
    else 0.0
)


# ================================================================
# 11. FINAL RESULTS
# ================================================================

optical_flow_results = pd.DataFrame(
    [
        {
            "Method":
                "Optical-Flow Propagation",

            "Period":
                OF_PERIOD,

            "mAP@0.5 (%)":
                of_ap50 * 100,

            "mAP@0.5:0.95 (%)":
                of_map5095 * 100,

            "Precision (%)":
                precision_of * 100,

            "Recall (%)":
                recall_of * 100,

            "F1 (%)":
                f1_of * 100,

            "Count MAE":
                count_mae_of,

            "Total Frames":
                n_total_of,

            "Detector Executions":
                n_detector_of,

            "Propagated Frames":
                n_propagated_of,

            "Frame Reduction (%)":
                frame_reduction_of * 100,

            "Detector Runtime (s)":
                detector_runtime_of,

            "Propagation Runtime (s)":
                propagation_runtime_of,

            "Mean Latency (ms)":
                mean_latency_of,

            "Proc. Rate (FPS)":
                processing_rate_of
        }
    ]
)


print("\n")
print("=" * 72)
print("OPTICAL-FLOW PROPAGATION FINAL RESULTS")
print("=" * 72)

print(
    optical_flow_results
    .T
    .to_string(
        header=False
    )
)


# ================================================================
# 12. SAVE
# ================================================================

of_eval[
    "schedule"
].to_csv(
    "optical_flow_schedule.csv",
    index=False
)

optical_flow_results.to_csv(
    "optical_flow_results.csv",
    index=False
)

print("\nSaved:")
print(" - optical_flow_schedule.csv")
print(" - optical_flow_results.csv")

Using the same held-out evaluation sequences as FrameHopper.
Evaluation sequences: 32
Evaluation frames: 44204

Running optical-flow propagation...

Optical-flow baseline: MVI_39501 (540 frames)
Optical-flow baseline: MVI_39511 (380 frames)
Optical-flow baseline: MVI_40701 (1130 frames)
Optical-flow baseline: MVI_40711 (1030 frames)
Optical-flow baseline: MVI_40712 (2400 frames)
Optical-flow baseline: MVI_40714 (1180 frames)
Optical-flow baseline: MVI_40742 (1655 frames)
Optical-flow baseline: MVI_40743 (1629 frames)
Optical-flow baseline: MVI_40761 (2030 frames)
Optical-flow baseline: MVI_40762 (1825 frames)
Optical-flow baseline: MVI_40763 (1741 frames)
Optical-flow baseline: MVI_40771 (1720 frames)
Optical-flow baseline: MVI_40772 (1200 frames)
Optical-flow baseline: MVI_40773 (985 frames)
Optical-flow baseline: MVI_40774 (950 frames)
Optical-flow baseline: MVI_40775 (975 frames)
Optical-flow baseline: MVI_40792 (1810 frames)
Optical-flow baseline: MVI_40793 (1959 frames)
Optical-fl